#Fruit Image Classifier — CNN Model

**Objective:** Build a Convolutional Neural Network (CNN) to classify images of **Apple**, **Orange**, and **Banana**.

**Pipeline Overview:**
1. Dataset Exploration & Visualisation
2. Data Cleaning — fix mis-labelled images
3. Class Balancing
4. Image Augmentation
5. CNN Model (Baseline)
6. CNN Model (Improved with Transfer Learning — MobileNetV2)
7. Evaluation & Results
8. Experiment Summary

---
## 0. Imports & Configuration

In [ ]:
import os
import sys
import shutil
import random
import math
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter
from PIL import Image
import urllib.request
import io

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Sklearn metrics
from sklearn.metrics import classification_report, confusion_matrix

# ── Paths & Configuration ────────────────────────────────────────────────────
GITHUB_BASE_URL = 'https://raw.githubusercontent.com/Codemelia/ml_ca/master'
TRAIN_IMAGES_PATH = f'{GITHUB_BASE_URL}/train'
TEST_IMAGES_PATH = f'{GITHUB_BASE_URL}/test'

# Create local temp directories
LOCAL_TRAIN_DIR = 'temp_train_data'
LOCAL_TEST_DIR = 'temp_test_data'
os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)
os.makedirs(LOCAL_TEST_DIR, exist_ok=True)

CLASSES = ['apple', 'banana', 'orange']
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Pretty plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 12
})

print('TensorFlow version:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))
print('Working with GitHub-sourced images...')

---
## 1. Download Images from GitHub

In [ ]:
def download_images_from_github(source_url, target_dir, classes):
    """
    Download images from GitHub and organize by class.
    """
    print(f'Downloading images from {source_url}...')
    
    counts = {cls: 0 for cls in classes}
    failed_images = []
    
    # Create class directories
    for cls in classes:
        os.makedirs(os.path.join(target_dir, cls), exist_ok=True)
    
    # Try to download each class's images
    for cls in classes:
        for i in range(1, 100):  # Try images 1-99
            filename = f'{cls}_{i}.jpg'
            url = f'{source_url}/{filename}'
            
            try:
                response = urllib.request.urlopen(url, timeout=5)
                img_data = response.read()
                
                # Verify it's a valid image
                img = Image.open(io.BytesIO(img_data))
                
                # Save to local directory
                save_path = os.path.join(target_dir, cls, filename)
                with open(save_path, 'wb') as f:
                    f.write(img_data)
                counts[cls] += 1
                
            except Exception as e:
                # Image doesn't exist or error downloading
                pass
    
    print(f'\n✓ Downloaded from {target_dir}:')
    for cls, count in counts.items():
        print(f'   {cls:10s}: {count} images')
    print(f'   {"TOTAL":10s}: {sum(counts.values())} images')
    
    return counts

# Download training and test images
train_counts = download_images_from_github(TRAIN_IMAGES_PATH, LOCAL_TRAIN_DIR, CLASSES)
test_counts = download_images_from_github(TEST_IMAGES_PATH, LOCAL_TEST_DIR, CLASSES)

---
## 2. Dataset Exploration

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ('red', 'orange', 'yellow')

for ax, (counts, title) in zip(axes, [(train_counts, 'Training Set'), (test_counts, 'Test Set')]):
    classes_list = list(counts.keys())
    values = list(counts.values())
    bars = ax.bar(classes_list, values, color=colors[:len(classes_list)], edgecolor='black', width=0.5)
    ax.set_title(f'Class Distribution — {title}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Number of Images')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', va='bottom', fontweight='bold')
    ax.set_ylim(0, max(values) * 1.15)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: class_distribution.png')

---
## 3. Load and Prepare Data for Training

In [ ]:
def load_images_from_directory(directory, img_size=(128, 128)):
    """
    Load all images from directory organized by class folders.
    Returns: (images_array, labels_array, label_to_class_map)
    """
    images = []
    labels = []
    label_to_class = {}
    class_counter = 0
    
    for class_folder in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, class_folder)
        if not os.path.isdir(class_path):
            continue
        
        label_to_class[class_counter] = class_folder
        
        for img_file in os.listdir(class_path):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(class_path, img_file)
                try:
                    img = Image.open(img_path).convert('RGB')
                    img = img.resize(img_size)
                    img_array = np.array(img) / 255.0
                    
                    images.append(img_array)
                    labels.append(class_counter)
                except Exception as e:
                    print(f'Error loading {img_path}: {e}')
        
        class_counter += 1
    
    return np.array(images), np.array(labels), label_to_class

# Load training and test data
X_train, y_train, label_map = load_images_from_directory(LOCAL_TRAIN_DIR)
X_test, y_test, _ = load_images_from_directory(LOCAL_TEST_DIR)

print(f'Training data shape: {X_train.shape}')
print(f'Training labels shape: {y_train.shape}')
print(f'Test data shape: {X_test.shape}')
print(f'Test labels shape: {y_test.shape}')
print(f'Label map: {label_map}')

# Convert labels to one-hot encoding
num_classes = len(label_map)
y_train_one_hot = keras.utils.to_categorical(y_train, num_classes)
y_test_one_hot = keras.utils.to_categorical(y_test, num_classes)

---
## 4. Split Training Data into Train/Validation

In [ ]:
from sklearn.model_selection import train_test_split

# Split training data: 80% train, 20% validation
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train_one_hot, test_size=0.2, random_state=SEED, stratify=y_train
)

print(f'Final Training set: {X_train_split.shape}')
print(f'Validation set: {X_val.shape}')
print(f'Test set: {X_test.shape}')

---
## 5. Model 1: Baseline CNN (WITHOUT Augmentation)

In [ ]:
def build_baseline_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation='relu'),
        
        layers.Flatten(),
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Train baseline model WITHOUT augmentation
print('Training Baseline CNN (NO Augmentation)...')
baseline_model = build_baseline_cnn(X_train_split.shape[1:], num_classes)
baseline_model.summary()

baseline_history = baseline_model.fit(
    X_train_split, y_train_split,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=BATCH_SIZE,
    verbose=1
)

# Evaluate on test set
baseline_test_loss, baseline_test_acc = baseline_model.evaluate(X_test, y_test_one_hot, verbose=0)
print(f'\nBaseline Model - Test Accuracy (NO Augmentation): {baseline_test_acc:.4f}')

---
## 6. Model 2: CNN WITH Data Augmentation

In [ ]:
# Create augmentation pipeline
augmentation_model = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2)
])

# Train model WITH augmentation
print('Training CNN WITH Data Augmentation...')
augmented_model = build_baseline_cnn(X_train_split.shape[1:], num_classes)

# Create augmented dataset
def augment_dataset(X, y, augmentation_model, num_augmentations=2):
    """
    Augment dataset by applying transformations.
    Returns: (augmented_X, augmented_y)
    """
    X_augmented = [X]
    y_augmented = [y]
    
    for _ in range(num_augmentations):
        X_aug = augmentation_model(X, training=True)
        X_augmented.append(X_aug.numpy())
        y_augmented.append(y)
    
    return np.concatenate(X_augmented), np.concatenate(y_augmented)

# Augment training data
X_train_augmented, y_train_augmented = augment_dataset(X_train_split, y_train_split, augmentation_model, num_augmentations=1)

augmented_history = augmented_model.fit(
    X_train_augmented, y_train_augmented,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=BATCH_SIZE,
    verbose=1
)

# Evaluate on test set (WITHOUT augmentation during evaluation)
augmented_test_loss, augmented_test_acc = augmented_model.evaluate(X_test, y_test_one_hot, verbose=0)
print(f'\nAugmented Model - Test Accuracy (WITH Augmentation in training): {augmented_test_acc:.4f}')

---
## 7. Compare Results

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Baseline - Accuracy
axes[0, 0].plot(baseline_history.history['accuracy'], label='Train', linewidth=2)
axes[0, 0].plot(baseline_history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0, 0].axhline(y=baseline_test_acc, color='r', linestyle='--', label=f'Test: {baseline_test_acc:.4f}')
axes[0, 0].set_title('Baseline CNN (NO Augmentation) - Accuracy', fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Baseline - Loss
axes[0, 1].plot(baseline_history.history['loss'], label='Train', linewidth=2)
axes[0, 1].plot(baseline_history.history['val_loss'], label='Validation', linewidth=2)
axes[0, 1].set_title('Baseline CNN (NO Augmentation) - Loss', fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Augmented - Accuracy
axes[1, 0].plot(augmented_history.history['accuracy'], label='Train', linewidth=2)
axes[1, 0].plot(augmented_history.history['val_accuracy'], label='Validation', linewidth=2)
axes[1, 0].axhline(y=augmented_test_acc, color='r', linestyle='--', label=f'Test: {augmented_test_acc:.4f}')
axes[1, 0].set_title('CNN WITH Data Augmentation - Accuracy', fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Augmented - Loss
axes[1, 1].plot(augmented_history.history['loss'], label='Train', linewidth=2)
axes[1, 1].plot(augmented_history.history['val_loss'], label='Validation', linewidth=2)
axes[1, 1].set_title('CNN WITH Data Augmentation - Loss', fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary comparison
print('\n' + '='*60)
print('RESULTS SUMMARY')
print('='*60)
print(f'\nBaseline CNN (NO Augmentation):')
print(f'  Final Validation Accuracy: {baseline_history.history["val_accuracy"][-1]:.4f}')
print(f'  Test Accuracy: {baseline_test_acc:.4f}')
print(f'\nCNN WITH Data Augmentation:')
print(f'  Final Validation Accuracy: {augmented_history.history["val_accuracy"][-1]:.4f}')
print(f'  Test Accuracy: {augmented_test_acc:.4f}')
print(f'\nImprovement: {(augmented_test_acc - baseline_test_acc)*100:.2f}%')
print('='*60)

---
## 8. Model Predictions & Confusion Matrix

In [ ]:
# Get predictions from augmented model
y_pred_probs = augmented_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_one_hot, axis=1)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[label_map[i] for i in range(num_classes)],
            yticklabels=[label_map[i] for i in range(num_classes)],
            ax=ax, cbar_kws={'label': 'Count'})
ax.set_title('Confusion Matrix - Augmented Model', fontweight='bold', fontsize=14)
ax.set_xlabel('Predicted', fontweight='bold')
ax.set_ylabel('Actual', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification Report
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=[label_map[i] for i in range(num_classes)]))

---
## 9. Clean Up Temporary Files

In [ ]:
# Optional: Clean up temporary directories after analysis
print('Removing temporary data directories...')
shutil.rmtree(LOCAL_TRAIN_DIR, ignore_errors=True)
shutil.rmtree(LOCAL_TEST_DIR, ignore_errors=True)
print('✓ Cleanup complete')